# Resumable Agent Human Questions

> **Time:** ~10-20 minutes | **Level:** Advanced

This notebook exercises the resumable extraction agent and the Python client's human-question methods end to end for **user playbook generation**:

1. Enable the resumable `ask_human` path for playbook extraction.
2. Publish an interaction that should create a pending human question.
3. List and fetch the question with `list_pending_tool_calls` and `get_pending_tool_call`.
4. Resolve the question with `resolve_pending_tool_call(..., answer=...)` and verify playbook generation resumes.
5. Edit the resolved answer with `update_pending_tool_call_answer` and verify another resume.

### Prerequisites
- Reflexio server running: `uv run reflexio services start`
- A real LLM provider key available to the backend process
- `REFLEXIO_URL` and `REFLEXIO_API_KEY` available in `.env` or the shell

In [4]:
# Import notebook dependencies and create a client bound to the local Reflexio server.
import json
import os
import sys
import uuid
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from IPython.display import Markdown, display
from rich import print as rprint
from rich.console import Console
from rich.json import JSON as RichJSON

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

try:
    from _display_helpers import *
except ModuleNotFoundError as exc:
    if exc.name != "pandas":
        raise
    console = Console(force_jupyter=True)

    def load_env() -> tuple[str, str]:
        load_dotenv(PROJECT_ROOT / ".env")
        url = os.environ.get(
            "REFLEXIO_URL",
            f"http://localhost:{os.environ.get('BACKEND_PORT', '8081')}",
        )
        api_key = os.environ.get("REFLEXIO_API_KEY", "")
        show_success(f"Environment loaded - server URL: {url}")
        return url, api_key

    def show_success(msg: str) -> None:
        rprint(f"[bold green]OK[/bold green] {msg}")

    def show_json(data: Any, title: str | None = None) -> None:
        if title:
            display(Markdown(f"### {title}"))
        if hasattr(data, "model_dump"):
            data = data.model_dump(mode="json")
        console.print(RichJSON(json.dumps(data, indent=2, default=str)))

    def show_response(response: Any, title: str | None = None) -> None:
        show_json(response, title)

    def show_playbooks(playbooks: list[Any], title: str = "Playbooks") -> None:
        display(Markdown(f"### {title}"))
        if not playbooks:
            rprint("[dim italic]No playbooks found.[/dim italic]")
            return
        for index, playbook in enumerate(playbooks, 1):
            rprint(f"[bold]{index}.[/bold] {getattr(playbook, 'content', '')}")
            trigger = getattr(playbook, "trigger", None)
            rationale = getattr(playbook, "rationale", None)
            if trigger:
                rprint(f"   [dim]Trigger:[/dim] {trigger}")
            if rationale:
                rprint(f"   [dim]Rationale:[/dim] {rationale}")

from notebook_helpers import (
    as_dict,
    dedupe_playbooks,
    playbook_text,
    question_mentions_marker,
    resumable_playbook_interactions,
    wait_for,
)
from reflexio import ReflexioClient

url, api_key = load_env()
client = ReflexioClient(url_endpoint=url, api_key=api_key, timeout=300)

RUN_ID = uuid.uuid4().hex[:8]
RUN_MARKER = f"NOTEBOOK_RESUMABLE_PLAYBOOK_{RUN_ID}"
USER_ID = f"resumable_playbook_user_{RUN_ID}"
AGENT_VERSION = f"resumable-human-question-{RUN_ID}"
SESSION_ID = f"resumable-human-question-{RUN_ID}"
SOURCE = "notebook_resumable_human_questions"
QUESTION_TEXT = (
    f"For {RUN_MARKER}, what is the canonical billing-dispute escalation path "
    "the agent should follow?"
)
FIRST_ANSWER = (
    "Escalate billing disputes to the Tier 2 Billing Queue after confirming "
    f"account ownership. Marker: {RUN_MARKER}"
)
EDITED_ANSWER = (
    "Escalate billing disputes to the Senior Billing Escalations queue after "
    f"confirming account ownership and checking invoice history. Marker: {RUN_MARKER}"
)

show_success(f"Run marker: {RUN_MARKER}")
rprint(f"User ID: {USER_ID}")
rprint(f"Agent version: {AGENT_VERSION}")

OK Environment loaded - server URL: http://localhost:8081

OK Run marker: NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747

User ID: resumable_playbook_user_fc6b6747

Agent version: resumable-human-question-fc6b6747

## Helpers

Tiny non-client helpers live in `notebooks/notebook_helpers.py` for polling, model-to-dict conversion, marker matching, and the fixed interaction payload. The Reflexio API calls stay inline below so each cell shows which `ReflexioClient` method to use.

## Configure The Resumable Playbook Path

This demo temporarily patches org configuration. The final cleanup cell restores these fields and invalidates the server cache.

In [6]:
# Read the current org config so the notebook can restore it during cleanup.
original_config = client.get_config()
original_config_patch = {
    "agent_context_prompt": original_config.agent_context_prompt,
    "profile_extractor_config": (
        original_config.profile_extractor_config.model_dump(mode="json")
        if original_config.profile_extractor_config
        else None
    ),
    "user_playbook_extractor_config": (
        original_config.user_playbook_extractor_config.model_dump(mode="json")
        if original_config.user_playbook_extractor_config
        else None
    ),
    "pending_tool_call_config": original_config.pending_tool_call_config.model_dump(mode="json"),
    "skip_should_run_check": original_config.skip_should_run_check,
    "window_size": original_config.window_size,
    "stride_size": original_config.stride_size,
}

playbook_prompt = f"""
This notebook validates resumable user-playbook extraction for marker {RUN_MARKER}.

When a conversation contains {RUN_MARKER} and the transcript does NOT explicitly
provide the canonical billing-dispute escalation path, call the ask_human tool
exactly once before finalizing. The ask_human question argument must be exactly:
{QUESTION_TEXT}
Use answer_format "one short escalation procedure" and tags ["{RUN_MARKER}", "billing-escalation"].

If no resolved human answer for {RUN_MARKER} is available in Prior Knowledge yet,
finish with exactly {{"playbooks": []}} after the ask_human tool call.

If a resolved human answer for {RUN_MARKER} is available in Prior Knowledge,
do not call ask_human again. Finish with exactly one playbook. The playbook
trigger, content, and rationale must all include {RUN_MARKER}. The content must
include the exact human-provided answer and must not invent any steps beyond the
answer. The playbook should teach future support agents what to do for billing
disputes when escalation is needed.
""".strip()

# Patch only the fields needed for this demo: playbook extraction + pending questions.
patch_response = client.update_config(
    {
        "agent_context_prompt": (
            "Customer support agent for billing, account, and escalation workflows."
        ),
        "profile_extractor_config": None,
        "user_playbook_extractor_config": {
            "extractor_name": "resumable_human_question_playbook_demo",
            "extraction_definition_prompt": playbook_prompt,
            "context_prompt": "Notebook demo for resumable playbook extraction.",
            "aggregation_config": {"min_cluster_size": 1},
            "window_size_override": 4,
            "stride_size_override": 4,
        },
        "pending_tool_call_config": {
            "enabled": True,
            "pending_ttl_seconds": 3600,
            "dedup_cache_seconds": 30,
            "prior_answer_valid_seconds": 3600,
            "resume_poll_interval_seconds": 1.0,
            "tool_overrides": {
                "ask_human": {
                    "pending_ttl_seconds": 3600,
                    "dedup_cache_seconds": 30,
                    "prior_answer_valid_seconds": 3600,
                }
            },
        },
        "skip_should_run_check": True,
        "window_size": 4,
        "stride_size": 4,
    }
)
assert patch_response.get("success"), patch_response
# Invalidate server-side cached config so subsequent publish calls see the patch.
client.invalidate_cache()

# Fetch the config again to verify the server accepted the human-question setup.
updated_config = client.get_config()
assert updated_config.pending_tool_call_config.enabled is True
assert updated_config.profile_extractor_config is None
assert updated_config.user_playbook_extractor_config is not None
assert updated_config.user_playbook_extractor_config.extractor_name == "resumable_human_question_playbook_demo"
show_success("Configured resumable playbook extraction with ask_human enabled")

OK Configured resumable playbook extraction with ask_human enabled

## Publish A Conversation That Requires Human Input

The interaction intentionally withholds the canonical escalation path. The playbook extractor should ask a human instead of guessing.

In [7]:
# Publish a short interaction that should make the playbook extractor call ask_human.
publish_response = client.publish_interaction(
    user_id=USER_ID,
    session_id=SESSION_ID,
    source=f"{SOURCE}_main",
    agent_version=AGENT_VERSION,
    wait_for_response=True,
    force_extraction=True,
    skip_aggregation=True,
    override_learning_stall=True,
    interactions=resumable_playbook_interactions(RUN_MARKER),
)
assert publish_response.success, publish_response
show_response(publish_response, "Publish response")

### Publish response

{
  "success": true,
  "message": "Interaction published successfully",
  "warnings": [],
  "request_id": "0c7bb111-8a23-487a-8fe8-7f3192d1a6ad",
  "endpoint_url": null,
  "storage_type": "sqlite",
  "storage_label": "<default sqlite>",
  "profiles_added": 0,
  "profiles_updated": null,
  "playbooks_added": 0,
  "playbooks_updated": null
}

## Find The Pending Human Question

This cell uses `list_pending_tool_calls(status="pending")` and filters to this run marker.

In [8]:
pending_question = wait_for(
    "the generated ask_human pending question",
    lambda: next(
        (
            as_dict(question)
            # List pending tool calls and pick the ask_human row for this run.
            for question in client.list_pending_tool_calls(
                status="pending", limit=100
            ).pending_tool_calls
            if question.tool_name == "ask_human"
            and question.user_id == USER_ID
            and question_mentions_marker(as_dict(question), RUN_MARKER)
        ),
        None,
    ),
    timeout_seconds=240,
    interval_seconds=5,
)

assert pending_question["status"] == "pending"
assert pending_question["tool_name"] == "ask_human"
assert pending_question["user_id"] == USER_ID
assert pending_question["question_text"] == QUESTION_TEXT
show_json(pending_question, "Pending question")

### Pending question

{
  "id": "ptc_2e959cecba274f2e81b025ebe8e488df",
  "org_id": "self-host-org",
  "scope": {
    "org_id": "self-host-org",
    "scope_kind": "org"
  },
  "scope_hash": "26f485ae4aeac42a79bc0e197f8cb678269446fd65b54add7548b98b38b3bef5",
  "tool_name": "ask_human",
  "dedup_key": "5f2b9dc180c970b327326cfde294715901f562fb4093c2425236916ff0f4b304",
  "status": "pending",
  "question_text": "For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, what is the canonical billing-dispute escalation path
the agent should follow?",
  "args": {
    "question": "For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, what is the canonical billing-dispute escalation path 
the agent should follow?",
    "answer_format": "one short escalation procedure"
  },
  "tags": [
    "NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747",
    "billing-escalation"
  ],
  "user_id": "resumable_playbook_user_fc6b6747",
  "answer_format": "one short escalation procedure",
  "result": null,
  "superseded_by": null,
  "created_at": "2026-06-22T21:39:57.370000Z",
  "resolved_at": null,
  "expires_at": "2026-06-22T22:39:57.339276Z",
  "cache_until": "2026-06-22T21:40:27.339276Z",
  "valid_until": null
}

## Fetch The Question By ID

`get_pending_tool_call` returns the same schema as the list endpoint for one known question.

In [9]:
# Fetch the same pending question by ID for detail-view style workflows.
fetched_question = as_dict(client.get_pending_tool_call(pending_question["id"]))

assert fetched_question["id"] == pending_question["id"]
assert fetched_question["status"] == "pending"
show_json(fetched_question, "Fetched pending question")

### Fetched pending question

{
  "id": "ptc_2e959cecba274f2e81b025ebe8e488df",
  "org_id": "self-host-org",
  "scope": {
    "org_id": "self-host-org",
    "scope_kind": "org"
  },
  "scope_hash": "26f485ae4aeac42a79bc0e197f8cb678269446fd65b54add7548b98b38b3bef5",
  "tool_name": "ask_human",
  "dedup_key": "5f2b9dc180c970b327326cfde294715901f562fb4093c2425236916ff0f4b304",
  "status": "pending",
  "question_text": "For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, what is the canonical billing-dispute escalation path
the agent should follow?",
  "args": {
    "question": "For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, what is the canonical billing-dispute escalation path 
the agent should follow?",
    "answer_format": "one short escalation procedure"
  },
  "tags": [
    "NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747",
    "billing-escalation"
  ],
  "user_id": "resumable_playbook_user_fc6b6747",
  "answer_format": "one short escalation procedure",
  "result": null,
  "superseded_by": null,
  "created_at": "2026-06-22T21:39:57.370000Z",
  "resolved_at": null,
  "expires_at": "2026-06-22T22:39:57.339276Z",
  "cache_until": "2026-06-22T21:40:27.339276Z",
  "valid_until": null
}

## Resolve The Question And Verify Resume

The `answer` convenience argument wraps the string as `{"answer": ...}`. The API schedules a resumable extraction drain, and the notebook polls until a user playbook appears with the human answer.

In [10]:
# Answer the human question; the client wraps answer=... as {"answer": ...}.
resolved_question = as_dict(
    client.resolve_pending_tool_call(
        pending_question["id"],
        answer=FIRST_ANSWER,
        valid_for_seconds=3600,
    )
)

assert resolved_question["status"] == "resolved"
assert resolved_question["result"] == {"answer": FIRST_ANSWER}
show_json(resolved_question, "Resolved question")

first_playbooks = wait_for(
    "a user playbook containing the first human answer",
    lambda: [
        playbook
        for playbook in dedupe_playbooks(
            [
                # First check the exact user/version playbook list.
                *client.get_user_playbooks(
                    user_id=USER_ID,
                    agent_version=AGENT_VERSION,
                    limit=50,
                ).user_playbooks,
                # Also search by answer text; useful when a local DB has older org-scoped resume work.
                *client.search_user_playbooks(
                    query="Tier 2 Billing Queue",
                    top_k=50,
                    threshold=0.0,
                ).user_playbooks,
            ]
        )
        if RUN_MARKER in playbook_text(playbook)
        and "Tier 2 Billing Queue" in playbook_text(playbook)
    ],
    timeout_seconds=300,
    interval_seconds=5,
)
show_playbooks(first_playbooks, "Playbooks after first answer resume")

### Resolved question

{
  "id": "ptc_2e959cecba274f2e81b025ebe8e488df",
  "org_id": "self-host-org",
  "scope": {
    "org_id": "self-host-org",
    "scope_kind": "org"
  },
  "scope_hash": "26f485ae4aeac42a79bc0e197f8cb678269446fd65b54add7548b98b38b3bef5",
  "tool_name": "ask_human",
  "dedup_key": "5f2b9dc180c970b327326cfde294715901f562fb4093c2425236916ff0f4b304",
  "status": "resolved",
  "question_text": "For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, what is the canonical billing-dispute escalation path
the agent should follow?",
  "args": {
    "question": "For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, what is the canonical billing-dispute escalation path 
the agent should follow?",
    "answer_format": "one short escalation procedure"
  },
  "tags": [
    "NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747",
    "billing-escalation"
  ],
  "user_id": "resumable_playbook_user_fc6b6747",
  "answer_format": "one short escalation procedure",
  "result": {
    "answer": "Escalate billing disputes to the Tier 2 Billing Queue after confirming account ownership. Marker: 
NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747"
  },
  "superseded_by": null,
  "created_at": "2026-06-22T21:39:57.370000Z",
  "resolved_at": "2026-06-22T21:40:01.499728Z",
  "expires_at": "2026-06-22T22:39:57.339276Z",
  "cache_until": "2026-06-22T21:40:27.339276Z",
  "valid_until": "2026-06-22T22:40:01.499728Z"
}

### Playbooks after first answer resume

1. For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, the canonical billing-dispute escalation path is: Escalate billing 
disputes to the Tier 2 Billing Queue after confirming account ownership.

Trigger: When a customer raises a billing dispute under NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747 and standard 
self-service resolution is insufficient or the customer requests escalation.

Rationale: Provides the canonical billing-dispute escalation path for NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, 
supplied by a human model developer, so future support agents follow the same verified procedure instead of 
guessing.

## Edit The Resolved Answer And Verify Re-Resume

Editing the answer marks dependent runs ready again. The notebook polls for a playbook that reflects the edited answer.

In [11]:
# Edit the resolved answer; Reflexio schedules another resume for dependent runs.
edited_question = as_dict(
    client.update_pending_tool_call_answer(
        pending_question["id"],
        EDITED_ANSWER,
        valid_for_seconds=3600,
    )
)

assert edited_question["status"] == "resolved"
assert edited_question["result"] == {"answer": EDITED_ANSWER}
show_json(edited_question, "Edited question")

edited_playbooks = wait_for(
    "a user playbook containing the edited human answer",
    lambda: [
        playbook
        for playbook in dedupe_playbooks(
            [
                # First check the exact user/version playbook list.
                *client.get_user_playbooks(
                    user_id=USER_ID,
                    agent_version=AGENT_VERSION,
                    limit=50,
                ).user_playbooks,
                # Search for the edited answer to confirm the re-resumed playbook reflects it.
                *client.search_user_playbooks(
                    query="Senior Billing Escalations",
                    top_k=50,
                    threshold=0.0,
                ).user_playbooks,
            ]
        )
        if RUN_MARKER in playbook_text(playbook)
        and "Senior Billing Escalations" in playbook_text(playbook)
    ],
    timeout_seconds=300,
    interval_seconds=5,
)
show_playbooks(edited_playbooks, "Playbooks after edited answer resume")

### Edited question

{
  "id": "ptc_2e959cecba274f2e81b025ebe8e488df",
  "org_id": "self-host-org",
  "scope": {
    "org_id": "self-host-org",
    "scope_kind": "org"
  },
  "scope_hash": "26f485ae4aeac42a79bc0e197f8cb678269446fd65b54add7548b98b38b3bef5",
  "tool_name": "ask_human",
  "dedup_key": "5f2b9dc180c970b327326cfde294715901f562fb4093c2425236916ff0f4b304",
  "status": "resolved",
  "question_text": "For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, what is the canonical billing-dispute escalation path
the agent should follow?",
  "args": {
    "question": "For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, what is the canonical billing-dispute escalation path 
the agent should follow?",
    "answer_format": "one short escalation procedure"
  },
  "tags": [
    "NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747",
    "billing-escalation"
  ],
  "user_id": "resumable_playbook_user_fc6b6747",
  "answer_format": "one short escalation procedure",
  "result": {
    "answer": "Escalate billing disputes to the Senior Billing Escalations queue after confirming account ownership
and checking invoice history. Marker: NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747"
  },
  "superseded_by": null,
  "created_at": "2026-06-22T21:39:57.370000Z",
  "resolved_at": "2026-06-22T21:40:16.790886Z",
  "expires_at": "2026-06-22T22:39:57.339276Z",
  "cache_until": "2026-06-22T21:40:27.339276Z",
  "valid_until": "2026-06-22T22:40:16.790886Z"
}

### Playbooks after edited answer resume

1. For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, the canonical billing-dispute escalation path is: Escalate billing 
disputes to the Senior Billing Escalations queue after confirming account ownership and checking invoice history.

Trigger: When a customer raises a billing dispute under NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747 and standard 
self-service resolution is insufficient or the customer requests escalation.

Rationale: Provides the canonical billing-dispute escalation path for NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, 
supplied by a human model developer, so future support agents follow the same verified procedure (confirm account 
ownership, check invoice history, then escalate to the Senior Billing Escalations queue) instead of guessing.

## Client-Side Resolve Validation

The client enforces that callers provide exactly one of `answer` or `result`.

In [12]:
for kwargs in ({}, {"answer": "x", "result": {"answer": "y"}}):
    try:
        # The SDK rejects ambiguous resolve calls before sending an HTTP request.
        client.resolve_pending_tool_call(pending_question["id"], **kwargs)
    except ValueError as exc:
        show_success(f"Got expected ValueError for {kwargs}: {exc}")
    else:
        raise AssertionError(f"Expected ValueError for {kwargs}")

OK Got expected ValueError for {}: Provide exactly one of 'result' or 'answer' to resolve.

OK Got expected ValueError for {'answer': 'x', 'result': {'answer': 'y'}}: Provide exactly one of 'result' or 
'answer' to resolve.

## Optional: Resolve With A Raw Result Payload

Set `RUN_RAW_RESULT_STEP = True` to create a fresh pending question and resolve it with the lower-level `result` payload.

In [13]:
RUN_RAW_RESULT_STEP = False

if RUN_RAW_RESULT_STEP:
    raw_user_id = f"{USER_ID}_raw_result"
    raw_session_id = f"{SESSION_ID}-raw-result"
    # Create an independent pending question so the optional raw-result demo is isolated.
    raw_publish_response = client.publish_interaction(
        user_id=raw_user_id,
        session_id=raw_session_id,
        source=f"{SOURCE}_raw_result",
        agent_version=AGENT_VERSION,
        wait_for_response=True,
        force_extraction=True,
        skip_aggregation=True,
        override_learning_stall=True,
        interactions=resumable_playbook_interactions(RUN_MARKER),
    )
    assert raw_publish_response.success, raw_publish_response

    raw_pending = wait_for(
        "a fresh pending question for raw-result resolution",
        lambda: next(
            (
                as_dict(question)
                for question in client.list_pending_tool_calls(
                    status="pending", limit=100
                ).pending_tool_calls
                if question.tool_name == "ask_human"
                and question.user_id == raw_user_id
                and question_mentions_marker(as_dict(question), RUN_MARKER)
            ),
            None,
        ),
        timeout_seconds=240,
        interval_seconds=5,
    )
    raw_result = {
        "answer": f"Route billing disputes to the Finance Operations queue. Marker: {RUN_MARKER}",
        "confidence": 0.9,
        "source": "notebook raw-result example",
    }
    # Use result=... when a tool returns a structured payload instead of plain answer text.
    raw_resolved = as_dict(
        client.resolve_pending_tool_call(
            raw_pending["id"],
            result=raw_result,
            valid_for_seconds=3600,
        )
    )
    assert raw_resolved["status"] == "resolved"
    assert raw_resolved["result"] == raw_result
    show_json(raw_resolved, "Question resolved with raw result")
else:
    show_success("Skipped raw-result step; set RUN_RAW_RESULT_STEP = True to run it")

OK Skipped raw-result step; set RUN_RAW_RESULT_STEP = True to run it

## Optional: Cancel A Pending Question

Set `RUN_CANCEL_STEP = True` to create a fresh pending question and cancel it. Cancellation only applies while the question is still pending.

In [14]:
RUN_CANCEL_STEP = False

if RUN_CANCEL_STEP:
    cancel_user_id = f"{USER_ID}_cancel"
    cancel_session_id = f"{SESSION_ID}-cancel"
    # Create an independent pending question so cancellation does not affect the main run.
    cancel_publish_response = client.publish_interaction(
        user_id=cancel_user_id,
        session_id=cancel_session_id,
        source=f"{SOURCE}_cancel",
        agent_version=AGENT_VERSION,
        wait_for_response=True,
        force_extraction=True,
        skip_aggregation=True,
        override_learning_stall=True,
        interactions=resumable_playbook_interactions(RUN_MARKER),
    )
    assert cancel_publish_response.success, cancel_publish_response

    cancel_pending = wait_for(
        "a fresh pending question for cancellation",
        lambda: next(
            (
                as_dict(question)
                for question in client.list_pending_tool_calls(
                    status="pending", limit=100
                ).pending_tool_calls
                if question.tool_name == "ask_human"
                and question.user_id == cancel_user_id
                and question_mentions_marker(as_dict(question), RUN_MARKER)
            ),
            None,
        ),
        timeout_seconds=240,
        interval_seconds=5,
    )
    # Cancel only applies while the ask_human row is still pending.
    cancelled_question = as_dict(client.cancel_pending_tool_call(cancel_pending["id"]))
    assert cancelled_question["status"] == "cancelled"
    show_json(cancelled_question, "Cancelled pending question")
else:
    show_success("Skipped cancel step; set RUN_CANCEL_STEP = True to run it")

OK Skipped cancel step; set RUN_CANCEL_STEP = True to run it

## Optional: Mark N/A

Set `RUN_MARK_NA_STEP = True` to create a fresh pending question and resolve it with the standard not-applicable result.

In [15]:
RUN_MARK_NA_STEP = False

if RUN_MARK_NA_STEP:
    na_user_id = f"{USER_ID}_not_applicable"
    na_session_id = f"{SESSION_ID}-not-applicable"
    # Create an independent pending question for the not-applicable flow.
    na_publish_response = client.publish_interaction(
        user_id=na_user_id,
        session_id=na_session_id,
        source=f"{SOURCE}_not_applicable",
        agent_version=AGENT_VERSION,
        wait_for_response=True,
        force_extraction=True,
        skip_aggregation=True,
        override_learning_stall=True,
        interactions=resumable_playbook_interactions(RUN_MARKER),
    )
    assert na_publish_response.success, na_publish_response

    na_pending = wait_for(
        "a fresh pending question for not-applicable resolution",
        lambda: next(
            (
                as_dict(question)
                for question in client.list_pending_tool_calls(
                    status="pending", limit=100
                ).pending_tool_calls
                if question.tool_name == "ask_human"
                and question.user_id == na_user_id
                and question_mentions_marker(as_dict(question), RUN_MARKER)
            ),
            None,
        ),
        timeout_seconds=240,
        interval_seconds=5,
    )
    # Mark N/A stores the standard no-information answer and unblocks dependent runs.
    na_question = as_dict(
        client.mark_pending_tool_call_not_applicable(
            na_pending["id"],
            valid_for_seconds=3600,
        )
    )
    assert na_question["status"] == "resolved"
    assert na_question["result"] == {
        "answer": "User does not have information about this question.",
        "not_applicable": True,
    }
    show_json(na_question, "Question marked N/A")
else:
    show_success("Skipped Mark N/A step; set RUN_MARK_NA_STEP = True to run it")

OK Skipped Mark N/A step; set RUN_MARK_NA_STEP = True to run it

## Inspect Final State

The main question should now be resolved with the edited answer, and generated user playbooks should remain available for review.

In [16]:
final_question = wait_for(
    "the run question in resolved state",
    lambda: next(
        (
            as_dict(question)
            # List resolved tool calls to show the final state of the main ask_human row.
            for question in client.list_pending_tool_calls(
                status="resolved", limit=100
            ).pending_tool_calls
            if question.tool_name == "ask_human"
            and question.user_id == USER_ID
            and question_mentions_marker(as_dict(question), RUN_MARKER)
        ),
        None,
    ),
    timeout_seconds=60,
    interval_seconds=3,
)
# Combine exact list + search results for a compact final view of marker playbooks.
final_playbooks = dedupe_playbooks(
    [
        *client.get_user_playbooks(
            user_id=USER_ID,
            agent_version=AGENT_VERSION,
            limit=50,
        ).user_playbooks,
        *client.search_user_playbooks(
            query=RUN_MARKER,
            top_k=50,
            threshold=0.0,
        ).user_playbooks,
    ]
)
final_playbooks = [
    playbook for playbook in final_playbooks if RUN_MARKER in playbook_text(playbook)
]

show_json(final_question, "Final question row")
show_playbooks(final_playbooks, "Final user playbooks")

### Final question row

{
  "id": "ptc_2e959cecba274f2e81b025ebe8e488df",
  "org_id": "self-host-org",
  "scope": {
    "org_id": "self-host-org",
    "scope_kind": "org"
  },
  "scope_hash": "26f485ae4aeac42a79bc0e197f8cb678269446fd65b54add7548b98b38b3bef5",
  "tool_name": "ask_human",
  "dedup_key": "5f2b9dc180c970b327326cfde294715901f562fb4093c2425236916ff0f4b304",
  "status": "resolved",
  "question_text": "For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, what is the canonical billing-dispute escalation path
the agent should follow?",
  "args": {
    "question": "For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, what is the canonical billing-dispute escalation path 
the agent should follow?",
    "answer_format": "one short escalation procedure"
  },
  "tags": [
    "NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747",
    "billing-escalation"
  ],
  "user_id": "resumable_playbook_user_fc6b6747",
  "answer_format": "one short escalation procedure",
  "result": {
    "answer": "Escalate billing disputes to the Senior Billing Escalations queue after confirming account ownership
and checking invoice history. Marker: NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747"
  },
  "superseded_by": null,
  "created_at": "2026-06-22T21:39:57.370000Z",
  "resolved_at": "2026-06-22T21:40:16.790886Z",
  "expires_at": "2026-06-22T22:39:57.339276Z",
  "cache_until": "2026-06-22T21:40:27.339276Z",
  "valid_until": "2026-06-22T22:40:16.790886Z"
}

### Final user playbooks

1. For NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, the canonical billing-dispute escalation path is: Escalate billing 
disputes to the Senior Billing Escalations queue after confirming account ownership and checking invoice history.

Trigger: When a customer raises a billing dispute under NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747 and standard 
self-service resolution is insufficient or the customer requests escalation.

Rationale: Provides the canonical billing-dispute escalation path for NOTEBOOK_RESUMABLE_PLAYBOOK_fc6b6747, 
supplied by a human model developer, so future support agents follow the same verified procedure (confirm account 
ownership, check invoice history, then escalate to the Senior Billing Escalations queue) instead of guessing.

## Cleanup

The default cleanup restores the org config fields changed by this notebook and invalidates the server cache. It does not delete generated interactions, questions, or playbooks.

In [17]:
RESTORE_ORIGINAL_CONFIG = True

if RESTORE_ORIGINAL_CONFIG:
    # Restore the config fields captured at the start of the notebook.
    restore_response = client.update_config(original_config_patch)
    assert restore_response.get("success"), restore_response
    # Clear cached config so later notebook/API calls see the restored settings.
    client.invalidate_cache()
    show_success("Restored original config fields")
else:
    show_success("Skipped config restore so the demo config remains inspectable")

OK Restored original config fields